In [ ]:
import pandas as pd
import numpy as np
from time import perf_counter
from datasets import load_dataset
from memory_profiler import memory_usage
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import BertTokenizer, BertModel
from sklearn.preprocessing import MultiLabelBinarizer


In [14]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PRETRAINED_MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128
MAX_EPOCHS = 4  # Maximum epochs for early stopping
PATIENCE = 3     # Patience for early stopping

tokenizer = BertTokenizer.from_pretrained(PRETRAINED_MODEL_NAME)

print(f"Using device: {DEVICE}")

Using device: cuda


In [ ]:
ds = load_dataset("TimSchopf/arxiv_categories", "default")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df

,review_text,ENTREGA,OUTROS,PRODUTO,CONDICOESDERECEBIMENTO,INADEQUADA,ANUNCIO
0,"Aparelho muito bom, confiável e com valor aqui...",0,0,1,0,0,0
1,"A história é muito boa, porém o autor ""enrolou...",0,0,1,0,0,0
2,"Entrega rápida, produto muito bom Amei. Pratic...",1,0,1,0,0,0
3,Produto otimo so falta o carregador da maquina...,0,0,1,1,0,0
4,a proteção anti queda não é boa se cair de fr...,0,0,1,0,0,0
...,...,...,...,...,...,...,...
7997,amei o produto. chegou no prazo e em perfeito ...,1,0,1,1,0,0
7998,Ótima embalagem. Produto entregue no prazo. Re...,1,0,1,1,0,0
7999,"ótimo produto, super recomendo .,Entrega bem r...",1,0,1,0,0,0
8000,"Veio tudo certinho, dentro do prazo e o produt...",1,0,1,1,0,0


In [ ]:
train_df = train_df.rename(columns={'title': 'text'})
val_df = val_df.rename(columns={'title': 'text'})
test_df = test_df.rename(columns={'title': 'text'})

train_df = train_df.rename(columns={'categories': 'labels'})
val_df = val_df.rename(columns={'categories': 'labels'})
test_df = test_df.rename(columns={'categories': 'labels'})

In [ ]:
allowed_categories = ["cs.AI", "cs.CL", "stat.ML", "math.OC", "cs.LG"]

def clean_element(lst):
    final = []
    for elem in lst:
        clean = elem.split('->')[-1]
        final.append(clean)
    return final

train_df['labels'] = train_df['labels'].apply(clean_element)
val_df['labels'] = val_df['labels'].apply(clean_element)
test_df['labels'] = test_df['labels'].apply(clean_element)

In [ ]:
train_df = train_df[train_df['labels'].apply(lambda cats: all(c in allowed_categories for c in cats))]
test_df = test_df[test_df['labels'].apply(lambda cats: all(c in allowed_categories for c in cats))]
val_df = val_df[val_df['labels'].apply(lambda cats: all(c in allowed_categories for c in cats))]

train_df = train_df[train_df['labels'].apply(len) > 0]
test_df = test_df[test_df['labels'].apply(len) > 0]
val_df = val_df[val_df['labels'].apply(len) > 0]

In [ ]:
train_df.drop(columns=['id','abstract','creation_date'], inplace=True)
test_df.drop(columns=['id','abstract','creation_date'], inplace=True)
val_df.drop(columns=['id','abstract','creation_date'], inplace=True)

train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)
val_df.reset_index(drop=True, inplace=True)

train_df

In [ ]:
mlb = MultiLabelBinarizer()

train_labels_binarized = mlb.fit_transform(train_df['labels'])
val_labels_binarized = mlb.transform(val_df['labels'])
test_labels_binarized = mlb.transform(test_df['labels'])

train_labels_df = pd.DataFrame(train_labels_binarized, columns=mlb.classes_)
val_labels_df = pd.DataFrame(val_labels_binarized, columns=mlb.classes_)
test_labels_df = pd.DataFrame(test_labels_binarized, columns=mlb.classes_)

train_df = pd.concat([train_df, train_labels_df], axis=1)
val_df = pd.concat([val_df, val_labels_df], axis=1)
test_df = pd.concat([test_df, test_labels_df], axis=1)

train_df = train_df.drop(columns=['labels'])
val_df = val_df.drop(columns=['labels'])
test_df = test_df.drop(columns=['labels'])

train_df

In [5]:
class MultiLabelClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        """
        Args:
            texts: List or array of text samples
            labels: 2D array of shape (num_samples, num_classes) with binary indicators (0 or 1)
            tokenizer: Pretrained tokenizer (e.g., BertTokenizer)
            max_len: Maximum sequence length
        """
        self.texts = texts
        self.labels = labels  # Shape: (num_samples, num_classes)
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]  # Shape: (num_classes,)
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.float)  # Binary vector for multilabel
        }

In [6]:
class BertForMultiLabelClassification(nn.Module):
    def __init__(self, num_classes):
        super(BertForMultiLabelClassification, self).__init__()
        self.bert = BertModel.from_pretrained(PRETRAINED_MODEL_NAME)
        self.pre_classifier = nn.Linear(768, 768)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(768, num_classes)  # Output logits for each class
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = outputs[0][:, 0]  # CLS token
        pooled_output = self.pre_classifier(hidden_state)
        pooled_output = nn.ReLU()(pooled_output)
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits  # Return raw logits for BCEWithLogitsLoss

In [7]:
def get_metrics(y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)

    precisions, recalls, f1s, supports = precision_recall_fscore_support(y_true, y_pred)

    return acc, precisions, recalls, f1s

In [8]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, save_path, max_epochs=MAX_EPOCHS, patience=PATIENCE):
    best_val_loss = float('inf')
    epochs_no_improve = 0
    start_train = perf_counter()
    
    # Initialize best metrics
    best_train_acc = 0
    best_train_precisions = None
    best_train_recalls = None
    best_train_f1s = None
    best_val_acc = 0
    best_val_precisions = None
    best_val_recalls = None
    best_val_f1s = None
    
    for epoch in range(max_epochs):
        model.train()
        train_loss = 0
        train_preds = []
        train_true = []
        
        for batch in tqdm(train_dataloader, desc=f'Epoch {epoch + 1}/{max_epochs}', leave=False):
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)  # Shape: (batch_size, num_classes)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)  # Shape: (batch_size, num_classes)
            loss = criterion(outputs, labels)  # BCEWithLogitsLoss
            train_loss += loss.item()
            # Compute binary predictions for each class
            preds = (torch.sigmoid(outputs) > 0.5).float().cpu().numpy()  # Shape: (batch_size, num_classes)
            train_preds.extend(preds)
            train_true.extend(labels.cpu().numpy())
            loss.backward()
            optimizer.step()
        
        train_loss /= len(train_dataloader)
        train_true = np.array(train_true)  # Shape: (num_samples, num_classes)
        train_preds = np.array(train_preds)  # Shape: (num_samples, num_classes)
        train_acc, train_precisions, train_recalls, train_f1s = get_metrics(train_true, train_preds)
        
        model.eval()
        val_loss = 0
        val_preds = []
        val_true = []
        with torch.no_grad():
            for batch in val_dataloader:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                preds = (torch.sigmoid(outputs) > 0.5).float().cpu().numpy()
                val_preds.extend(preds)
                val_true.extend(labels.cpu().numpy())
        
        val_loss /= len(val_dataloader)
        val_true = np.array(val_true)  # Shape: (num_samples, num_classes)
        val_preds = np.array(val_preds)  # Shape: (num_samples, num_classes)
        val_acc, val_precisions, val_recalls, val_f1s = get_metrics(val_true, val_preds)
        
        print(f"Epoch {epoch + 1}/{max_epochs} - Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1s}")
        print(f"Epoch {epoch + 1}/{max_epochs} - Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1s}")
        
        # Early stopping logic
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_train_acc = train_acc
            best_train_precisions = train_precisions
            best_train_recalls = train_recalls
            best_train_f1s = train_f1s
            best_val_acc = val_acc
            best_val_precisions = val_precisions
            best_val_recalls = val_recalls
            best_val_f1s = val_f1s
            torch.save(model.state_dict(), save_path)
            epochs_no_improve = 0
            print("Model saved!")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered")
                break
    
    total_train_time = perf_counter() - start_train
    return (best_train_acc, best_train_precisions, best_train_recalls, best_train_f1s,
            best_val_acc, best_val_precisions, best_val_recalls, best_val_f1s, total_train_time)

In [9]:
def evaluate_model(model, test_dataloader):
    model.eval()
    predictions = []
    true_labels = []
    classification_times = []
    
    start_test = perf_counter()
    
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Testing"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)  # Shape: (batch_size, num_classes)
            
            for i in range(input_ids.size(0)):
                input_id = input_ids[i].unsqueeze(0)
                attention_mask_sample = attention_mask[i].unsqueeze(0)
                label = labels[i].cpu().numpy()  # Shape: (num_classes,)
                
                start_time = perf_counter()
                
                output = model(input_ids=input_id, attention_mask=attention_mask_sample)  # Shape: (1, num_classes)
                pred = (torch.sigmoid(output) > 0.5).float().cpu().numpy()[0]  # Shape: (num_classes,)
                
                predictions.append(pred)
                true_labels.append(label)
                classification_times.append(perf_counter() - start_time)
    
    total_test_time = perf_counter() - start_test
    print(f"Test Time: {total_test_time:.2f} seconds")
    
    predictions = np.array(predictions)  # Shape: (num_samples, num_classes)
    true_labels = np.array(true_labels)  # Shape: (num_samples, num_classes)
    
    acc, precisions, recalls, f1s = get_metrics(true_labels, predictions)
    
    print("Test Metrics:")
    print("Accuracy:", acc)
    print("F1s:", f1s)
    print("Precisions:", precisions)
    print("Recalls:", recalls)
    
    return predictions, true_labels

In [ ]:
train_texts = train_df['text'].values
train_labels = train_df.drop(columns=['text']).values

val_texts = val_df['text'].values
val_labels = val_df.drop(columns=['text']).values

test_texts = test_df['text'].values
test_labels = test_df.drop(columns=['text']).values

num_classes = train_labels.shape[1]

# Create datasets
train_dataset = MultiLabelClassificationDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = MultiLabelClassificationDataset(val_texts, val_labels, tokenizer, MAX_LEN)
test_dataset = MultiLabelClassificationDataset(test_texts, test_labels, tokenizer, MAX_LEN)

seeds = [2, 3, 5]
batch_sizes = [16, 32]
learning_rates = [5e-5, 3e-5, 2e-5]
results = []

# Grid search loop
for batch_size in batch_sizes:
    for learning_rate in learning_rates:
        for seed in seeds:
            torch.manual_seed(seed)
            model = BertForMultiLabelClassification(num_classes).to(DEVICE)
            optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
            criterion = nn.BCEWithLogitsLoss()  # For multi-label classification
            
            train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
            val_dataloader = DataLoader(val_dataset, batch_size=batch_size)
            test_dataloader = DataLoader(test_dataset, batch_size=batch_size)
            
            save_path = f'results/bert_multilabel3_bs{batch_size}_lr{learning_rate}_seed{seed}.pt'

            # Train
            if torch.cuda.is_available():
                torch.cuda.reset_max_memory_allocated()

            max_memory_usage_train, retval = memory_usage(
                (train_model, (model, train_dataloader, val_dataloader, optimizer, criterion, save_path),
                 {'max_epochs': MAX_EPOCHS, 'patience': PATIENCE}), max_usage=True, retval=True)
            
            max_vram_usage_train = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

            (train_acc, train_precisions, train_recalls, train_f1s,
             val_acc, val_precisions, val_recalls, val_f1s, total_train_time) = retval
            
            # Load best model
            model.load_state_dict(torch.load(save_path))
            
            # Evaluate
            if torch.cuda.is_available():
                torch.cuda.reset_max_memory_allocated()

            start = perf_counter()
            max_memory_usage_test, test_retval = memory_usage(
                (evaluate_model, (model, test_dataloader), {}), max_usage=True, retval=True)
            total_time_test = perf_counter() - start

            max_vram_usage_test = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

            predictions, true_labels = test_retval
            test_acc, test_precisions, test_recalls, test_f1s = get_metrics(true_labels, predictions)
            
            # Store individual seed results
            results.append({
                'batch_size': batch_size,
                'learning_rate': learning_rate,
                'seed': seed,
                'train_acc': train_acc,
                'train_precisions': train_precisions.tolist(),
                'train_recalls': train_recalls.tolist(),
                'train_f1s': train_f1s.tolist(),
                'max_memory_usage_train': max_memory_usage_train,
                'max_vram_usage_train': max_vram_usage_train,
                'total_train_time': total_train_time,
                'val_acc': val_acc,
                'val_precisions': val_precisions.tolist(),
                'val_recalls': val_recalls.tolist(),
                'val_f1s': val_f1s.tolist(),
                'test_acc': test_acc,
                'test_precisions': test_precisions.tolist(),
                'test_recalls': test_recalls.tolist(),
                'test_f1s': test_f1s.tolist(),
                'max_memory_usage_test': max_memory_usage_test,
                'max_vram_usage_test': max_vram_usage_test,
                'total_test_time': total_time_test
            })

In [ ]:
df = pd.DataFrame(results)
df.to_csv('results/bert_multilabel3.csv', index=False)

In [12]:
df

,batch_size,learning_rate,seed,train_acc,train_precisions,train_recalls,train_f1s,max_memory_usage_train,max_vram_usage_train,total_train_time,...,val_precisions,val_recalls,val_f1s,test_acc,test_precisions,test_recalls,test_f1s,max_memory_usage_test,max_vram_usage_test,total_test_time
0,16,0.00005,2,0.461538,"[0.0, 0.0, 0.7948717948717948, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8857142857142857, 0.0, 0.0]",1248.843750,2528.906738,5.273854,...,"[0.0, 0.0, 0.7, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8235294117647058, 0.0, 0.0]",0.6,"[0.0, 0.0, 0.9, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.9473684210526315, 0.0, 0.0]",1189.027344,1715.291016,1.259908
1,16,0.00005,3,0.461538,"[0.0, 0.0, 0.7948717948717948, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8857142857142857, 0.0, 0.0]",1268.070312,2539.156738,8.143835,...,"[0.0, 0.0, 0.7, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8235294117647058, 0.0, 0.0]",0.6,"[0.0, 0.0, 0.9, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.9473684210526315, 0.0, 0.0]",1272.113281,1717.791016,1.203630
2,16,0.00005,5,0.461538,"[0.0, 0.0, 0.7948717948717948, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8857142857142857, 0.0, 0.0]",1268.578125,2542.031738,7.289995,...,"[0.0, 0.0, 0.7, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8235294117647058, 0.0, 0.0]",0.6,"[0.0, 0.0, 0.9, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.9473684210526315, 0.0, 0.0]",1272.488281,1717.916016,1.215830
3,16,0.00003,2,0.461538,"[0.0, 0.0, 0.7948717948717948, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8857142857142857, 0.0, 0.0]",1268.855469,2549.156738,8.758684,...,"[0.0, 0.0, 0.7, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8235294117647058, 0.0, 0.0]",0.6,"[0.0, 0.0, 0.9, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.9473684210526315, 0.0, 0.0]",1272.671875,1725.541016,1.369954
4,16,0.00003,3,0.461538,"[0.0, 0.0, 0.7948717948717948, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8857142857142857, 0.0, 0.0]",1268.972656,2547.031738,8.919644,...,"[0.0, 0.0, 0.7, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8235294117647058, 0.0, 0.0]",0.6,"[0.0, 0.0, 0.9, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.9473684210526315, 0.0, 0.0]",1272.789062,1723.416016,1.301652
5,16,0.00003,5,0.461538,"[0.0, 0.0, 0.7948717948717948, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8857142857142857, 0.0, 0.0]",1269.082031,2548.031738,8.887331,...,"[0.0, 0.0, 0.7, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8235294117647058, 0.0, 0.0]",0.6,"[0.0, 0.0, 0.9, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.9473684210526315, 0.0, 0.0]",1272.816406,1722.916016,1.300475
6,16,0.00002,2,0.461538,"[0.0, 0.0, 0.7948717948717948, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8857142857142857, 0.0, 0.0]",1269.085938,2546.156738,8.987676,...,"[0.0, 0.0, 0.7, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8235294117647058, 0.0, 0.0]",0.6,"[0.0, 0.0, 0.9, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.9473684210526315, 0.0, 0.0]",1272.820312,1721.541016,1.343361
7,16,0.00002,3,0.461538,"[0.0, 0.0, 0.7948717948717948, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8857142857142857, 0.0, 0.0]",1269.082031,2546.281738,9.135816,...,"[0.0, 0.0, 0.7, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8235294117647058, 0.0, 0.0]",0.6,"[0.0, 0.0, 0.9, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.9473684210526315, 0.0, 0.0]",1272.816406,1724.416016,1.336011
8,16,0.00002,5,0.461538,"[0.0, 0.0, 0.7948717948717948, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8857142857142857, 0.0, 0.0]",1269.089844,2542.406738,9.080358,...,"[0.0, 0.0, 0.7, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8235294117647058, 0.0, 0.0]",0.6,"[0.0, 0.0, 0.9, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.9473684210526315, 0.0, 0.0]",1272.828125,1715.791016,1.285718
9,32,0.00005,2,0.461538